<a href="https://colab.research.google.com/github/Bruno120715/Checkpoint-Merge/blob/main/CheckpointMerge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Configuración de Checkpoint Merge
Define los parámetros para descargar los modelos desde Civitai o enlaces externos, y configura las opciones de fusión (Merge Mode).

In [1]:
# @title Parámetros de Fusión y Descarga

# @markdown ### Autenticación
civitai_api_key = "9e9c9ea79b18b2b84220e29e479e86e1"  # @param {type:"string"}

# @markdown ### Enlaces de Modelos (Checkpoints)
checkpoint_link_1 = "https://civitai.com/api/download/models/3391532?fileId=3281025"  # @param {type:"string"}
checkpoint_link_2 = "https://civitai.com/api/download/models/274039?fileId=214158"  # @param {type:"string"}
checkpoint_link_3 = "https://civitai.com/api/download/models/4224?fileId=4121"  # @param {type:"string"}

# @markdown ### Configuración del Merge
base_model = "SD 1.5"  # @param ["SD 1.5", "SDXL"]
merge_mode = "Weighted Sum"  # @param ["Weighted Sum", "Add Difference", "Sigmoid", "Spline", "Beta"]
calculation_mode = "fp16"  # @param ["fp16", "fp32", "bf16", "fp8"]
fp16_conversion = True  # @param {type:"boolean"}
alpha = 0.5  # @param {type:"slider", min:0.0, max:1.0, step:0.05}

import os
import requests

def download_model(url, target_path, api_key=None):
    if not url:
        print("Enlace vacío, omitiendo...")
        return None

    # Si es un enlace de Civitai, añadir el token de API si existe
    if "civitai.com" in url and api_key:
        connector = "&" if "?" in url else "?"
        url = f"{url}{connector}token={api_key}"

    print(f"Descargando modelo desde: {url}")
    os.makedirs(os.path.dirname(target_path), exist_ok=True)

    response = requests.get(url, stream=True)
    if response.status_code == 200:
        with open(target_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
        print(f"Guardado en: {target_path}")
        return target_path
    else:
        print(f"Error al descargar: Código de estado {response.status_code}")
        return None

print("Configuración cargada correctamente.")

Configuración cargada correctamente.


In [2]:
# @title 1. Instalar dependencias necesarias
!pip install -q --upgrade safetensors torch pytorch-lightning

ERROR: Operation cancelled by user


KeyboardInterrupt: 

In [ ]:
# @title 2. Descargar Checkpoints y Ejecutar Fusión (Merge)
import os
import torch
import io
import pickle
from safetensors.torch import load_file, save_file

# Definir nombres locales para almacenar los archivos descargados
model_path_1 = "/content/model_1.safetensors"
model_path_2 = "/content/model_2.safetensors"
model_path_3 = "/content/model_3.safetensors"
output_path = "/content/merged_model.safetensors"

# 1. Descargar modelos usando las variables definidas en la configuración
print("--- Iniciando descargas de checkpoints ---")
path_1 = download_model(checkpoint_link_1, model_path_1, civitai_api_key) if checkpoint_link_1 else None
path_2 = download_model(checkpoint_link_2, model_path_2, civitai_api_key) if checkpoint_link_2 else None
path_3 = download_model(checkpoint_link_3, model_path_3, civitai_api_key) if checkpoint_link_3 else None

def safe_torch_load(filepath):
    """Carga un checkpoint de PyTorch interceptando clases de Lightning/Pydantic de manera segura mediante monkeypatch."""
    # Guardar referencia original del método de deserialización estándar
    original_find_class = pickle.Unpickler.find_class

    def patched_find_class(self, module, name):
        # Si intenta buscar clases conflictivas de pytorch_lightning o pydantic, devolver una clase Dummy vacía
        if any(x in module for x in ["pytorch_lightning", "lightning", "pydantic"]):
            class Dummy:
                def __init__(self, *args, **kwargs): pass
                def __setstate__(self, state): pass
            return Dummy
        try:
            return original_find_class(self, module, name)
        except Exception:
            class DummyFallback:
                def __init__(self, *args, **kwargs): pass
                def __setstate__(self, state): pass
            return DummyFallback

    # Aplicar el parche temporal en el módulo pickle importado globalmente
    pickle.Unpickler.find_class = patched_find_class

    try:
        print("-> Cargando archivo con deserializador seguro...")
        with open(filepath, 'rb') as f:
            buffer = io.BytesIO(f.read())
        # Se carga usando el módulo pickle estándar (parcheado temporalmente)
        checkpoint = torch.load(buffer, map_location='cpu')
    finally:
        # Garantizar que el método original sea restaurado siempre
        pickle.Unpickler.find_class = original_find_class

    return checkpoint

def load_model_weights(filepath):
    """Carga pesos de forma segura detectando si es PyTorch o Safetensors."""
    if not filepath or not os.path.exists(filepath):
        return None

    # Comprobar la firma del archivo (PK\x03\x04 indica un archivo zip/pytorch ckpt)
    with open(filepath, 'rb') as f:
        header = f.read(4)

    if header == b'PK\x03\x04':
        print(f"-> Detectado formato PyTorch (.ckpt) para {filepath}.")
        try:
            checkpoint = safe_torch_load(filepath)
        except Exception as e:
            print(f"Fallo crítico en la carga del archivo PyTorch: {e}")
            raise e

        # Los checkpoints de SD suelen tener los pesos bajo 'state_dict'
        if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
            return checkpoint['state_dict']
        return checkpoint
    else:
        print(f"-> Detectado formato Safetensors para {filepath}. Cargando con load_file...")
        return load_file(filepath)

def run_safetensors_merge():
    if not path_1 or not path_2:
        print("Error: Se requieren al menos el Checkpoint 1 y 2 para fusionar.")
        return

    print(f"\n--- Leyendo tensores de los archivos para modelo base {base_model} ---")
    dict_a = load_model_weights(path_1)
    dict_b = load_model_weights(path_2)
    dict_c = load_model_weights(path_3) if ((merge_mode == "Add Difference" or merge_mode == "Beta") and path_3) else None

    # Determinar tipo de dato de cálculo
    if calculation_mode == "fp16":
        dtype = torch.float16
    elif calculation_mode == "bf16":
        dtype = torch.bfloat16
    elif calculation_mode == "fp8":
        dtype = torch.float8_e4m3fn if hasattr(torch, 'float8_e4m3fn') else torch.float16
        if dtype == torch.float16:
            print("Advertencia: FP8 no está soportado en este entorno de PyTorch, usando FP16 como respaldo.")
    else:
        dtype = torch.float32

    merged_dict = {}
    print(f"Aplicando fusión: Mode = {merge_mode}, Alpha = {alpha}, Precision = {calculation_mode}")

    keys_a = set(dict_a.keys())
    keys_b = set(dict_b.keys())
    common_keys = keys_a.intersection(keys_b)

    print(f"Total de llaves compartidas encontradas: {len(common_keys)}")

    for key in common_keys:
        t_a = dict_a[key]
        t_b = dict_b[key]

        # Asegurarse de que ambos sean tensores de PyTorch antes de convertir
        if not isinstance(t_a, torch.Tensor) or not isinstance(t_b, torch.Tensor):
            continue

        t_a = t_a.to(dtype)
        t_b = t_b.to(dtype)

        if merge_mode == "Weighted Sum":
            merged_tensor = (1.0 - alpha) * t_a + alpha * t_b
        elif merge_mode == "Add Difference" and dict_c is not None and key in dict_c:
            t_c = dict_c[key]
            if isinstance(t_c, torch.Tensor):
                t_c = t_c.to(dtype)
                merged_tensor = t_a + alpha * (t_b - t_c)
            else:
                merged_tensor = t_a
        elif merge_mode == "Beta":
            # Modo de fusión Beta: Interpolación no lineal usando una curva de distribución Beta simplificada
            # Aplica un sesgo de transición suave basado en la intensidad de los pesos
            beta_weight = 3 * (alpha ** 2) - 2 * (alpha ** 3) # Suavizado Hermite (Smoothstep)
            merged_tensor = (1.0 - beta_weight) * t_a + beta_weight * t_b
        else:
            # Si no aplica otro modo, por defecto se asigna el tensor del modelo A
            merged_tensor = t_a

        # Forzar guardado en fp16 si fp16_conversion está activo
        if fp16_conversion:
            merged_tensor = merged_tensor.to(torch.float16)

        merged_dict[key] = merged_tensor

    # Copiar llaves únicas del modelo A que no estuvieran en B
    for key in keys_a - common_keys:
        t_a = dict_a[key]
        if isinstance(t_a, torch.Tensor):
            merged_dict[key] = t_a.to(torch.float16) if fp16_conversion else t_a

    print(f"Guardando resultado final en: {output_path}...")
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    save_file(merged_dict, output_path)
    print("¡El proceso de fusión ha finalizado de manera exitosa!")

try:
    run_safetensors_merge()
except Exception as e:
    print(f"Ocurrió un error inesperado durante el merge: {e}")

### Guardar Modelo en Google Drive
Usa las siguientes celdas para montar tu Google Drive y transferir el modelo resultante para guardarlo de manera permanente.

In [ ]:
# @title 3. Conectar a Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# @title 4. Copiar Modelo Fusionado a Google Drive
import shutil
import os

# @markdown Ruta de la carpeta de destino en tu Google Drive
drive_folder = "/content/drive/MyDrive/StableDiffusion/Models" # @param {type:"string"}

# @markdown Nombre personalizado para tu archivo final (sin extensión)
custom_model_name = "Broken_Dynasty_XL" # @param {type:"string"}

os.makedirs(drive_folder, exist_ok=True)

source_file = "/content/merged_model.safetensors"

# Asegurar que el nombre tenga la extensión correcta
filename = custom_model_name.strip()
if not filename.endswith(".safetensors"):
    filename += ".safetensors"

target_file = os.path.join(drive_folder, filename)

if os.path.exists(source_file):
    print(f"Copiando {source_file} a {target_file}...")
    shutil.copy(source_file, target_file)
    print("¡Modelo copiado con éxito a tu Google Drive!")
else:
    print("Error: El archivo de modelo fusionado no existe en /content/merged_model.safetensors. Asegúrate de ejecutar la celda de fusión primero.")

Copiando /content/merged_model.safetensors a /content/drive/MyDrive/StableDiffusion/Models/Broken_Dynasty_XL.safetensors...
¡Modelo copiado con éxito a tu Google Drive!


### Instalar Stable Diffusion WebUI Forge
Ejecuta la siguiente celda para clonar el repositorio de Forge en tu entorno de Colab.

In [ ]:
# @title 5. Clonar e Instalar SD WebUI Forge
%cd /content
!git clone https://github.com/lllyasviel/stable-diffusion-webui-forge.git

# Crear enlaces simbólicos para que Forge pueda usar el modelo fusionado directamente
merged_model_source = "/content/merged_model.safetensors"
forge_model_dir = "/content/stable-diffusion-webui-forge/models/Stable-diffusion"

import os
if os.path.exists(merged_model_source):
    os.makedirs(forge_model_dir, exist_ok=True)
    target_link = os.path.join(forge_model_dir, "merged_model.safetensors")
    if not os.path.exists(target_link):
        os.symlink(merged_model_source, target_link)
        print("¡Modelo fusionado vinculado correctamente a Forge!")
else:
    print("El modelo fusionado local no se encontró. Puedes copiarlo manualmente a la carpeta de modelos de Forge si lo deseas.")

/content
Cloning into 'stable-diffusion-webui-forge'...
remote: Enumerating objects: 41984, done.
remote: Total 41984 (delta 0), reused 0 (delta 0), pack-reused 41984 (from 1)
Receiving objects: 100% (41984/41984), 46.38 MiB | 9.00 MiB/s, done.
Resolving deltas: 100% (29153/29153), done.
¡Modelo fusionado vinculado correctamente a Forge!
